# Multi-class classification with different kinds of supervision

Iris flowers are classified into three classes, A, B and C. A standard classifier requires one label per example. The supervision can be any logical statement about the classes: here the same network is trained from two knowledge bases and the results are compared.

1. **Full supervision**: the class of every example.
2. **Ambiguous labels**: for every example, two classes it could belong to.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import sltn
from sltn import Signature, Interpretation, KB

torch.manual_seed(0)
np.random.seed(0)
device = "cpu"

## Data

The Iris dataset: 4 real features per flower, 120 flowers for training and 30 for testing. Labels 0, 1, 2 stand for the classes A, B, C.

In [ ]:
train_df = pd.read_csv(Path("../datasets/iris_training.csv"))
test_df = pd.read_csv(Path("../datasets/iris_test.csv"))
y_train = torch.tensor(train_df.pop("species").to_numpy()).long().to(device)
y_test = torch.tensor(test_df.pop("species").to_numpy()).long().to(device)
x_train = torch.tensor(train_df.to_numpy()).float().to(device)
x_test = torch.tensor(test_df.to_numpy()).float().to(device)
print(x_train.shape, x_test.shape, torch.bincount(y_train).tolist())

## 1 — Signature

- a sort `Item` for a flower;
- one predicate per class: `A`, `B`, `C`;
- variables: `x_A`, `x_B`, `x_C` hold the examples of one class; `x_AvB`, `x_BvC`, `x_AvC` the examples known to be in one of two classes; `x` all the examples.

In [ ]:
sig = Signature("iris_clf")
sig.sort("Item")
for P in ["A", "B", "C"]:
    sig.predicate(P, ["Item"])
for v in ["x_A", "x_B", "x_C", "x_AvB", "x_BvC", "x_AvC", "x"]:
    sig.variable(v, "Item")
print(sig)

## 2 — Knowledge bases

**Single label.** Every flower belongs to exactly one class. This is stated as axioms on all the examples `x`:

- *at least one class*: `∀ x: A(x) ∨ B(x) ∨ C(x)`;
- *at most one class*: `∀ x: ¬(A(x) ∧ B(x))`, and the same for B, C and A, C.

These four clauses are added to both knowledge bases below. (The quantifier scope is narrow, Tutorial 2, so the bodies are parenthesised.)

**Full supervision.** One clause per class: `∀ x_A: A(x_A)`, and the same for B and C.

**Ambiguous labels.** Each example is given as "A or B", "B or C" or "A or C": `∀ x_AvB: A(x_AvB) ∨ B(x_AvB)`, and so on. A flower of class A is grounded both in `x_AvB` and in `x_AvC`, so the network sees it as "A or B" and as "A or C"; with exclusivity, only A satisfies both. No single clause gives a class, but together they determine it.

In [ ]:
exclusive = [
    sig.parse("∀ x: (A(x) ∨ B(x) ∨ C(x))"),
    *[sig.parse(f"∀ x: ¬({P}(x) ∧ {Q}(x))") for P, Q in ["AB", "BC", "AC"]],
]
full = KB([sig.parse(f"∀ x_{P}: {P}(x_{P})") for P in "ABC"] + exclusive)
ambiguous = KB([sig.parse(f"∀ x_{P}v{Q}: ({P}(x_{P}v{Q}) ∨ {Q}(x_{P}v{Q}))") for P, Q in ["AB", "BC", "AC"]] + exclusive)
for name, kb in [("full", full), ("ambiguous", ambiguous)]:
    print(f"{name:10} {kb}")

## 3 — Model

One MLP with three sigmoid outputs gives the truth values of `A`, `B` and `C`, its three columns. The outputs are independent: the network itself allows a flower to be in several classes or in none, and only the axioms of Section 2 make the classes exclusive. Each predicate is a small `ClassHead` module wrapping the shared MLP, so that it is registered in the interpretation. `output` selects the last layer; the note after the results uses a softmax instead.

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, output=torch.nn.Sigmoid):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(4, 16), torch.nn.ELU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(16, 16), torch.nn.ELU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(16, 3), output() if output is torch.nn.Sigmoid else output(dim=-1),
        )

    def forward(self, x):
        return self.net(x)


class ClassHead(torch.nn.Module):
    # Column k of the shared MLP output
    def __init__(self, mlp, k):
        super().__init__()
        self.mlp, self.k = mlp, k

    def forward(self, x):
        return self.mlp(x)[..., [self.k]]

## 4 — Interpretation

A fresh interpretation (and network) for each knowledge base. `interp.parameters()` returns the shared MLP's weights once, even though three groundings contain it.

In [ ]:
def make_interp(output=torch.nn.Sigmoid):
    interp = Interpretation(sig).to(device)
    interp["Item"] = sltn.Type("Item", shape=(4,))
    mlp = MLP(output)
    for k, P in enumerate("ABC"):
        interp[P] = ClassHead(mlp, k)
    return interp

print(make_interp())

## 5 — Grounding the variables, and metrics

`ground` puts a batch into the variables: the examples of each class, the pairs of possible classes, and all the examples.

The **accuracy** takes the class with the largest truth value among `A(x)`, `B(x)`, `C(x)`.

To verify that the classes are exclusive, `exclusivity` returns the mean of $A+B+C$ over the examples (1 for a one-hot output) and the truth value of the four single-label axioms.

In [ ]:
def ground(interp, x, y):
    for k, P in enumerate("ABC"):
        interp[f"x_{P}"] = x[y == k]
    interp["x_AvB"] = x[y != 2]
    interp["x_BvC"] = x[y != 0]
    interp["x_AvC"] = x[y != 1]
    interp["x"] = x


def accuracy(interp, x, y):
    interp.eval()
    with torch.no_grad():
        interp["x"] = x
        scores = torch.cat([interp(f"{P}(x)").value for P in "ABC"], dim=1)
    return (scores.argmax(dim=1) == y).float().mean().item()


single_label = KB(exclusive)

def exclusivity(interp, x):
    interp.eval()
    with torch.no_grad():
        interp["x"] = x
        scores = torch.cat([interp(f"{P}(x)").value for P in "ABC"], dim=1)
        return scores.sum(dim=1).mean().item(), interp(single_label).value.item()

## 6 — Training

The loss is $1 - \mathrm{SatAgg}(\mathcal{K})$, on minibatches of 12 flowers. Dropout is on during training (`interp.train()`) and off for the metrics (`interp.eval()`, Tutorial 4).

In [ ]:
from sltn.interpretation.operators import AggregPMeanError, ImpliesGoguen

def train(kb, epochs, seed=0, output=torch.nn.Sigmoid, log=True):
    torch.manual_seed(seed)
    interp = make_interp(output)
    interp.logic["SatAgg"] = AggregPMeanError(p=2, stable=True)

    optimizer = torch.optim.Adam(interp.parameters(), lr=0.001)
    for epoch in range(epochs):
        interp.train()
        for idx in torch.randperm(len(x_train)).split(12):
            ground(interp, x_train[idx], y_train[idx])
            optimizer.zero_grad()
            loss = 1 - interp(kb).value
            loss.backward()
            optimizer.step()
        if log and (epoch % (epochs // 4) == 0 or epoch == epochs - 1):
            interp.eval()
            with torch.no_grad():
                ground(interp, x_train, y_train)
                sat = interp(kb).value.item()
            print(f"  epoch {epoch:5d} | sat {sat:.3f} | acc train {accuracy(interp, x_train, y_train):.3f} "
                  f"test {accuracy(interp, x_test, y_test):.3f}")
    return interp


models = {}
for name, kb, epochs in [("full", full, 500), ("ambiguous", ambiguous, 500)]:
    print(name)
    models[name] = train(kb, epochs)

## 7 — Results

In [ ]:
print(f"{'knowledge base':15} {'train acc':>9} {'test acc':>9} {'mean A+B+C':>11} {'sat single-label':>17}")
for name, interp in models.items():
    s, e = exclusivity(interp, x_test)
    print(f"{name:15} {accuracy(interp, x_train, y_train):9.3f} {accuracy(interp, x_test, y_test):9.3f} {s:11.3f} {e:17.3f}")

With full supervision and with ambiguous labels the network learns the classes, the latter without ever being given a class. The outputs are close to exclusive ($A+B+C \approx 1$): exclusivity is learned from the axioms, not built into the network.

### Note
The single-label axioms can also be hardwired: a softmax output makes $A+B+C=1$ exactly, for every input, so raising one class lowers the others. The axioms then hold by construction and can be dropped from the knowledge bases. Below, the same two knowledge bases without the four axioms, trained with a softmax MLP. For comparison, the sigmoid network without the axioms shows what happens when the constraint is stated nowhere.

In [ ]:
def without_axioms(kb):
    return KB([c for c in kb.clauses if all(c is not e for e in exclusive)])

print(f"{'knowledge base':12} {'model':19} {'train acc':>9} {'test acc':>9} {'mean A+B+C':>11} {'sat single-label':>17}")
for name, kb in [("full", full), ("ambiguous", ambiguous)]:
    variants = {
        "sigmoid, no axioms": train(without_axioms(kb), 500, log=False),
        "softmax, no axioms": train(without_axioms(kb), 500, output=torch.nn.Softmax, log=False),
    }
    for kind, interp in variants.items():
        s, e = exclusivity(interp, x_test)
        print(f"{name:12} {kind:19} {accuracy(interp, x_train, y_train):9.3f} {accuracy(interp, x_test, y_test):9.3f} {s:11.3f} {e:17.3f}")

Without axioms the sigmoid network makes every class true (A+B+C ≈ 3) and the accuracy collapses. The softmax reaches the accuracy of the axiom version: exclusivity holds exactly, by construction.

## Summary

- Supervision kinds: one label per example, or sets of possible classes.
- Labels are given by grounding one variable per class (or per set of classes) with the matching examples.
- "Single label" is itself knowledge: two kinds of axioms (at least one class, at most one class) make independent predicates mutually exclusive. Without them the network can satisfy the labels by making every class true.
- A shared network grounds several predicates through small `nn.Module` heads, so it is registered once and trained by every clause.
- A constraint can also be built into the model (a softmax for exclusivity). It then holds exactly and gives a stronger bias, but it is no longer visible, inspectable or switchable in the knowledge base.